# 🐳 PHASE 2: Docker, Netzwerk & Reverse Proxy (Traefik + OAuth2-Proxy)
**Projekt: Mai_AI (MaiOmni) — Reverse Proxy & Dynamic routing**

In dieser Phase schalten wir das schützende **Traefik-Gateway** und den **Google OAuth2 Proxy** vor dein System. Dies entspricht der Architektur-Spezifikation für einen sicheren Web-Zugang über DuckDNS (`mai-ai.duckdns.org`), ohne den Host angreifbar zu machen.

--- 
### 🎯 Ziele dieser Phase:
1. Validierung des lokalen Docker-Status.
2. Erstellung des isolierten Plattform-Netzwerks (`ai_platform_network`).
3. Initialisierung von Traefik und des Auth-Gateways über Docker-Compose.
4. Verifizierung des sicheren Header-Verhaltens (`X-Forwarded-User`).

---

### 🛠️ Schritt 1: Docker CLI & Daemon Konnektivität prüfen
Wir prüfen mittels Python Docker SDK, ob die Docker Engine läuft und erreichbar ist.

In [7]:
import docker
import os
import sys
import time
import subprocess

def connect_docker(retries=3, delay=5):
    """
    Überprüft robust, ob der Docker-Daemon läuft.
    Falls nicht, wird die offizielle Docker.app (macOS), Docker Desktop (Windows) 
    oder der Systemdienst gestartet.
    Die Info-Ausgaben werden im Jupyter Notebook (.ipynb) angezeigt.
    """
    for i in range(retries):
        try:
            # 1. Schritt: Versuchen, eine Verbindung zum laufenden Daemon herzustellen
            client = docker.from_env()
            client.ping()
            print(f"[✓] Docker-Daemon ist aktiv. Server Version: {client.version()['Version']}")
            return client
        except (docker.errors.DockerException, Exception) as e:
            print(f"[!] Docker-Daemon ist aktuell nicht erreichbar (Versuch {i+1}/{retries}).")
            
            if i < retries - 1:
                print(f"[...] Starte Docker-Umgebung offiziell über das System... Bitte warten.")
                try:
                    # Start-Logik je nach Betriebssystem
                    
                    if sys.platform == "darwin":  # macOS
                        # Der offizielle, vom System unterstützte Weg über das App-Bundle.
                        subprocess.run(["open", "-a", "Docker"], check=True)
                        
                    elif sys.platform == "win32":  # Windows
                        # Sicherer Pfad über die offizielle Executable, um UAC/Admin-Rechte-Probleme
                        # beim direkten Starten des Windows-Dienstes zu umgehen.
                        program_files = os.environ.get("ProgramFiles", "C:\\Program Files")
                        docker_win_path = os.path.join(program_files, "Docker", "Docker", "Docker Desktop.exe")
                        
                        if os.path.exists(docker_win_path):
                            subprocess.Popen(
                                [docker_win_path],
                                start_new_session=True
                            )
                        else:
                            # Fallback auf den Windows-Dienst, falls der Standardpfad abweicht
                            subprocess.Popen(
                                ["net", "start", "com.docker.service"],
                                creationflags=subprocess.CREATE_NO_WINDOW,
                                start_new_session=True
                            )
                        
                    elif sys.platform.startswith("linux"):  # Linux
                        # Da 'sudo systemctl' im Notebook ohne Passworteingabe einfrieren würde,
                        # geben wir eine klare Handlungsanweisung aus und nutzen die systemd Socket-Aktivierung.
                        print("[!] HINWEIS: Docker unter Linux ist offline.")
                        print("[!] Bitte starte den Dienst im Terminal mit: 'sudo systemctl start docker'")
                        print("[...] Versuche dennoch, eine Verbindung über Socket-Activation aufzubauen...")
                        
                except Exception as start_error:
                    print(f"[X] Fehler beim offiziellen Startaufruf von Docker: {start_error}")
                
                # Dem Daemon ausreichend Zeit geben, die virtuelle Umgebung hochzufahren
                print(f"[...] Warte {delay + 3} Sekunden, damit die Engine vollständig initialisieren kann...")
                time.sleep(delay + 3)
                print(f"[...] Überprüfe Verbindung nach Startversuch erneut...\n")
            else:
                print("\n[X] Docker konnte nicht gestartet werden. Bitte öffne Docker Desktop manuell.")
                raise e

# Skript ausführen
try:
    client = connect_docker()
except Exception as final_error:
    print(f"\nAbbruch: Verbindung zu Docker fehlgeschlagen.\nDetails: {final_error}")

[✓] Docker-Daemon ist aktiv. Server Version: 29.5.3


### 🌐 Schritt 2: Docker-Netzwerk initialisieren
Die dynamic user containers kommunizieren über das dedizierte Brücken-Netzwerk `ai_platform_network` mit Traefik.

Schritt 0 (Vorbereitung): Verbindung zu Docker herstellen (dein connect_docker()-Code von vorhin).

Schritt 1 (Kommunikation): Dein Netzwerk erstellen (das ist genau der Code, den du gepostet hast).

Schritt 2 (Ablage/Datenschnittstelle): Die Volumes (Speicherplätze) für die Container erstellen (z. B. für deine Datenbank- oder Modell-Daten).

Schritt 3 (Der Container): Die Container starten und sie fest mit dem Netzwerk aus Schritt 1 und den Volumes aus Schritt 2 verdrahten.

In [8]:
import docker
import sys

# 0. Verbindung zu Docker herstellen
try:
    client = docker.from_env()
    client.ping()
except Exception as e:
    print(f"[X] Docker läuft nicht oder ist nicht erreichbar: {e}")
    sys.exit(1)

print("-" * 50)

# ==========================================
# 1. DIE KOMMUNIKATION (Netzwerk-Prüfung)
# ==========================================
network_name = "mai-ai_network"
try:
    networks = client.networks.list(names=[network_name])
    if not networks:
        client.networks.create(network_name, driver="bridge", attachable=True)
        print(f"[✓] Netzwerk '{network_name}' wurde erfolgreich erstellt.")
    else:
        print(f"[✓] Netzwerk '{network_name}' ist bereits vorhanden.")
except Exception as e:
    print(f"[!] Fehler beim Erstellen des Netzwerks: {e}")

# ==========================================
# 2. DIE ABLAGE & DATENSCHNITTSTELLE (Volumes)
# ==========================================
volumes_to_check = ["mai_ai_local_models", "mai_ai_db_data", "mai_ai_config"]
for volume in volumes_to_check:
    try:
        client.volumes.get(volume)
        print(f"[✓] Datenschnittstelle '{volume}' ist vorhanden.")
    except docker.errors.NotFound:
        client.volumes.create(name=volume)
        print(f"[✓] Datenschnittstelle '{volume}' wurde neu angelegt.")
    except Exception as e:
        print(f"[!] Fehler bei der Datenschnittstelle {volume}: {e}")

# ==========================================
# 3. DIE CONTAINER (Status-Überprüfung)
# ==========================================
# Liste deiner geplanten Container-Namen für das System
required_containers = ["mai_ai_ollama_engine"]
for container_name in required_containers:
    try:
        container = client.containers.get(container_name)
        print(f"[✓] Container '{container_name}' existiert (Status: {container.status}).")
    except docker.errors.NotFound:
        print(f"[!] Container '{container_name}' fehlt noch (Wird im nächsten Schritt erzeugt).")
    except Exception as e:
        print(f"[!] Fehler bei der Container-Überprüfung {container_name}: {e}")

print("-" * 50)
print("[✓] Infrastruktur-Check abgeschlossen.")

--------------------------------------------------
[✓] Netzwerk 'mai-ai_network' ist bereits vorhanden.
[✓] Datenschnittstelle 'mai_ai_local_models' ist vorhanden.
[✓] Datenschnittstelle 'mai_ai_db_data' ist vorhanden.
[✓] Datenschnittstelle 'mai_ai_config' ist vorhanden.
[!] Container 'mai_ai_ollama_engine' fehlt noch (Wird im nächsten Schritt erzeugt).
--------------------------------------------------
[✓] Infrastruktur-Check abgeschlossen.


### 🏗️ Schritt 3: Infrastruktur-Gateway starten (Traefik & OAuth2-Proxy)
Jetzt rufen wir die systemeigene Start-Sequenz auf, um Traefik und Google OAuth2 Proxy hochzufahren. (Stelle sicher, dass du eine `.env` Datei basierend auf `.env.example` erstellt hast!)

In [10]:
import os
import subprocess
import logging

# Einfaches Logging für das Notebook-Feedback
print("\n" + "="*60)
print(" 🐳 PHASE 2: ORCHESTRIERUNG & START-VALIDIERUNG")
print("="*60 + "\n")

env_path = "../.env"

# 1. Logik-Prüfung: Existiert die Konfiguration?
if not os.path.exists(env_path):
    print("[!] KONTROLLE FEHLGESCHLAGEN:")
    print("    Keine .env-Datei im Projekt-Root gefunden!")
    print("    Bitte kopiere '.env.example' zu '.env' und trage deine OAuth2-Daten ein.")
else:
    print("[✓] Umgebungskonfiguration (.env) lokalisiert.")
    
    # Optionaler Pre-Flight-Check der Variablen für das Notebook-Protokoll
    try:
        with open(env_path, "r") as f:
            env_content = f.read()
            
        print("    -> Überprüfe Funktionsbereich 1 & 2 (Ingress & Security)...")
        if "GOOGLE_CLIENT_ID" in env_content and "DOMAIN_NAME" in env_content:
            print("    [✓] OAuth2-Zugangsdaten und Domain-Konfiguration sind hinterlegt.")
        else:
            print("    [!] Warnung: Einige Schlüsselvariablen fehlen möglicherweise in der .env!")
    except Exception as e:
        print(f"    [!] Fehler beim Lesen der .env-Metadaten: {e}")

    # 2. Logik-Prüfung: Start der 3 Funktionsbereiche via Start_AI.py
    print("\n[*] Initialisiere System-Start über den AI Orchestrator...")
    try:
        # Führt das überarbeitete Start-Skript aus, das Ingress, Security und Engine koppelt
        result = subprocess.run(["python", "../src/docker_py/Start_AI.py"], check=True)
        
        print("\n" + "="*60)
        print("[✓] ERFOLG: Start_AI.py hat die Infrastruktur übergeben.")
        print("    Alle Container (Traefik, Auth, Ollama) sind im 'mai-ai_network' aktiv.")
        print("="*60)
        
    except subprocess.CalledProcessError as e:
        print(f"\n[!] FEHLER: Start_AI.py wurde mit Fehlercode {e.returncode} abgebrochen.")
        print("    Bitte überprüfe die Docker Desktop Logs der Container.")
    except Exception as e:
        print(f"\n[!] Unerwarteter Fehler beim Skriptaufruf: {e}")


 🐳 PHASE 2: ORCHESTRIERUNG & START-VALIDIERUNG

[!] KONTROLLE FEHLGESCHLAGEN:
    Keine .env-Datei im Projekt-Root gefunden!
    Bitte kopiere '.env.example' zu '.env' und trage deine OAuth2-Daten ein.


### 🔄 Was kommt als Nächstes?
Deine Gateway-Infrastruktur läuft im Hintergrund und filtert unautorisierte Anfragen.

Fahre fort mit dem nächsten Knotenpunkt: 
👉 **[03_html_embed.ipynb](file:notebooks/03_html_embed.ipynb)** um das Streamlit-User-Image zu bauen und den Dynamic Provisioner zu testen.